# LeetCode 1464 — Maximum Product of Two Elements in an Array bằng Python

**Link đề:** https://leetcode.com/problems/maximum-product-of-two-elements-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Heap (Priority Queue)

```python
class Solution:
    def maxProduct(self, nums: list[int]) -> int:
        ...
```

### Mục tiêu học
1. Đọc kỹ đề: hai **chỉ số khác nhau** `i != j` (giá trị có thể bằng nhau).
2. Lập luận toán học vì sao đáp án là **hai số lớn nhất**: `(a-1)(b-1)` tăng theo từng biến khi `a, b >= 1`.
3. Đi qua 5 lời giải: hai vòng lặp (hint) → sắp xếp → hai lượt tìm max → **một lượt giữ `max1`, `max2`** → `heapq.nlargest`.
4. Trace việc cập nhật `max1/max2` (nhánh "đẩy xuống" và nhánh "chỉ thay hạng 2"), chứng minh invariant, Time/Space.
5. Biến thể: có số âm (*Maximum Product of Three Numbers*), *Maximum Product Difference Between Two Pairs*, top-k.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `nums`. Chọn **hai chỉ số khác nhau** `i` và `j`. Trả về giá trị lớn nhất của `(nums[i] - 1) * (nums[j] - 1)`.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---:|---|
| 1 | `[3,4,5,2]` | 12 | `(4-1)*(5-1) = 3*4` |
| 2 | `[1,5,4,5]` | 16 | `(5-1)*(5-1)` — hai số 5 ở **hai chỉ số khác nhau** |
| 3 | `[3,7]` | 12 | `(3-1)*(7-1) = 2*6` |

**Constraints**
- `2 <= nums.length <= 500`
- `1 <= nums[i] <= 10^3`

Câu hỏi gợi mở:
- Ví dụ 2: được dùng hai số `5` dù giá trị bằng nhau không? (Có — khác **chỉ số** là đủ.)
- Vì `nums[i] >= 1` nên `nums[i] - 1 >= 0`. Điều này giúp gì cho việc chọn số?

In [1]:
nums = [1, 5, 4, 5]
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        print(f"i={i}, j={j}: ({nums[i]}-1)*({nums[j]}-1) = {(nums[i]-1)*(nums[j]-1)}")

i=0, j=1: (1-1)*(5-1) = 0
i=0, j=2: (1-1)*(4-1) = 0
i=0, j=3: (1-1)*(5-1) = 0
i=1, j=2: (5-1)*(4-1) = 12
i=1, j=3: (5-1)*(5-1) = 16
i=2, j=3: (4-1)*(5-1) = 12


## 2. Nền tảng Python cần dùng

### 2.1. Hai vòng lặp cho mọi cặp `i < j`

`for j in range(i + 1, n)` bắt đầu từ `i + 1` → mỗi cặp được xét **đúng một lần** và không bao giờ `i == j`. Số cặp là `n(n-1)/2`.

In [2]:
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs)
print("số cặp:", len(pairs), "= n(n-1)/2 =", n * (n - 1) // 2)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số cặp: 6 = n(n-1)/2 = 6


### 2.2. `sorted`, chỉ số âm `[-1]`, `[-2]`

Sau khi sắp xếp tăng, `s[-1]` là phần tử lớn nhất, `s[-2]` lớn thứ hai (có thể **bằng** `s[-1]` nếu trùng giá trị).

In [3]:
s = sorted([1, 5, 4, 5])
print(s, "| s[-1] =", s[-1], "| s[-2] =", s[-2])

[1, 4, 5, 5] | s[-1] = 5 | s[-2] = 5


### 2.3. `max`, `list.index`, `list.remove`

- `max(a)` trả giá trị lớn nhất (`O(n)`).
- `a.index(v)` trả vị trí **đầu tiên** có giá trị `v`.
- `a.remove(v)` xóa **một** lần xuất hiện đầu tiên của `v` (sửa list tại chỗ).

In [4]:
a = [1, 5, 4, 5]
print(max(a), a.index(5))
c = a[:]            # sao chép trước khi sửa
c.remove(5)
print(c, "| max còn lại:", max(c), "| a gốc:", a)

5 1
[1, 4, 5] | max còn lại: 5 | a gốc: [1, 5, 4, 5]


### 2.4. `heapq.nlargest(k, a)`

Module `heapq` cài **heap** (đống nhị phân). `nlargest(k, a)` trả `k` phần tử lớn nhất theo thứ tự giảm dần, với chi phí `O(n log k)` — với `k = 2` gần như `O(n)`.

In [5]:
import heapq
print(heapq.nlargest(2, [3, 4, 5, 2]))   # [5, 4]
print(heapq.nlargest(2, [1, 5, 4, 5]))   # [5, 5]

[5, 4]
[5, 5]


### 2.5. Tính đơn điệu của tích khi các thừa số không âm

Đặt `x = a - 1 >= 0`, `y = b - 1 >= 0`. Nếu tăng `x` (giữ `y`), tích `x*y` **không giảm** (vì `y >= 0`). Vậy muốn tích lớn nhất, chọn hai `x` lớn nhất ⇔ hai `a` lớn nhất.

Cảnh báo: điều này **sai** khi có số âm — ví dụ `(-5)*(-6) = 30 > 2*3`. Đề này có `nums[i] >= 1` nên an toàn (xem biến thể 15.1).

In [6]:
y = 3
print([x * y for x in [0, 1, 2, 3, 4]])     # tăng dần khi y >= 0
y = -3
print([x * y for x in [0, 1, 2, 3, 4]])     # GIẢM dần khi y < 0

[0, 3, 6, 9, 12]
[0, -3, -6, -9, -12]


## 3. Cách 1 — Brute force hai vòng lặp (theo hint)

In [7]:
def max_product_brute(nums):
    best = 0
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            best = max(best, (nums[i] - 1) * (nums[j] - 1))
    return best

print(max_product_brute([3, 4, 5, 2]))   # 12
print(max_product_brute([1, 5, 4, 5]))   # 16
print(max_product_brute([3, 7]))         # 12

12
16
12


Khởi tạo `best = 0` an toàn vì mọi tích `>= 0` (mỗi thừa số `>= 0`). Nếu bài cho phép số âm, phải khởi tạo `float('-inf')` hoặc tích của cặp đầu tiên.

### 3.1. Chi phí

`n(n-1)/2` cặp → `O(n²)`. Với `n = 500` là 124.750 phép nhân — qua dễ dàng, nhưng ta đang thử những cặp **rõ ràng không thể thắng** (ví dụ cặp chứa số nhỏ nhất).

In [8]:
for n in [10, 100, 500]:
    print(f"n={n:>3}: {n * (n - 1) // 2} cặp")

n= 10: 45 cặp
n=100: 4950 cặp
n=500: 124750 cặp


## 4. Nút thắt và insight

**Nút thắt:** thử mọi cặp, trong khi theo mục 2.5 chỉ **một cặp** có thể thắng: hai số lớn nhất.

**Insight:** đáp án = `(max1 - 1) * (max2 - 1)` với `max1 >= max2` là hai giá trị lớn nhất **ở hai chỉ số khác nhau** (nếu giá trị lớn nhất xuất hiện 2 lần thì `max2 = max1`).

Kiểm tra trên ví dụ:

In [9]:
for nums in [[3, 4, 5, 2], [1, 5, 4, 5], [3, 7]]:
    s = sorted(nums)
    print(nums, "→ hai số lớn nhất", s[-1], s[-2], "→", (s[-1] - 1) * (s[-2] - 1), "| brute:", max_product_brute(nums))

[3, 4, 5, 2] → hai số lớn nhất 5 4 → 12 | brute: 12
[1, 5, 4, 5] → hai số lớn nhất 5 5 → 16 | brute: 16
[3, 7] → hai số lớn nhất 7 3 → 12 | brute: 12


## 5. Cách 2 — Sắp xếp, lấy hai phần tử cuối

In [10]:
def max_product_sort(nums):
    s = sorted(nums)
    return (s[-1] - 1) * (s[-2] - 1)

print(max_product_sort([3, 4, 5, 2]), max_product_sort([1, 5, 4, 5]), max_product_sort([3, 7]))

12 16 12


Gọn nhưng tốn `O(n log n)` để sắp xếp **toàn bộ** mảng, trong khi ta chỉ cần 2 phần tử.

## 6. Cách 3 — Hai lượt: tìm max, bỏ đúng **một** lần xuất hiện, tìm max lần nữa

In [11]:
def max_product_two_pass(nums):
    i1 = nums.index(max(nums))                         # chỉ số của max thứ nhất
    max2 = max(nums[k] for k in range(len(nums)) if k != i1)
    return (nums[i1] - 1) * (max2 - 1)

print(max_product_two_pass([3, 4, 5, 2]), max_product_two_pass([1, 5, 4, 5]), max_product_two_pass([3, 7]))

12 16 12


Loại theo **chỉ số** `k != i1`, không phải theo **giá trị** — nếu loại mọi phần tử bằng `max` thì ví dụ 2 mất cả hai số 5 (lỗi ở mục 13).

## 7. Cách 4 (lời giải chính) — Một lượt giữ `max1`, `max2`

### 7.1. Mổ xẻ

Giữ hai biến: `max1` = lớn nhất đã thấy, `max2` = lớn thứ hai đã thấy (ở chỉ số khác). Khởi tạo `0` (nhỏ hơn mọi phần tử vì `nums[i] >= 1`).

Với mỗi `x`:
```python
if x >= max1:            # x thành quán quân mới
    max2 = max1          #   quán quân cũ bị đẩy xuống hạng 2
    max1 = x
elif x > max2:           # x không vượt max1 nhưng vượt max2
    max2 = x             #   chỉ thay hạng 2
# ngược lại: x không vào top 2, bỏ qua
```

Thứ tự `max2 = max1` **trước** `max1 = x` rất quan trọng: làm ngược sẽ mất quán quân cũ.

Dùng `>=` ở nhánh đầu để khi gặp số **bằng** `max1` (ví dụ số 5 thứ hai), số cũ được đẩy xuống `max2` → `max2 = max1 = 5`. (Dùng `>` cũng đúng, vì khi đó nhánh `elif x > max2` sẽ bắt — xem test ở mục 12.)

In [12]:
def max_product_one_pass(nums):
    max1 = max2 = 0
    for x in nums:
        if x >= max1:
            max2 = max1
            max1 = x
        elif x > max2:
            max2 = x
    return (max1 - 1) * (max2 - 1)

print(max_product_one_pass([3, 4, 5, 2]), max_product_one_pass([1, 5, 4, 5]), max_product_one_pass([3, 7]))

12 16 12


Python còn cho viết gọn hai dòng đẩy xuống: `max1, max2 = x, max1` (vế phải được tính trước khi gán — giống phép hoán đổi).

## 8. TRACE từng bước — `nums = [3, 4, 5, 2]`

| x | `max1` trước | `max2` trước | nhánh | `max1` sau | `max2` sau |
|---:|---:|---:|---|---:|---:|
| 3 | 0 | 0 | `x >= max1` → đẩy xuống | 3 | 0 |
| 4 | 3 | 0 | `x >= max1` → đẩy xuống | 4 | 3 |
| 5 | 4 | 3 | `x >= max1` → đẩy xuống | 5 | 4 |
| 2 | 5 | 4 | không vào top 2 | 5 | 4 |

Đáp án `(5-1)*(4-1) = 12`.

In [13]:
def trace_max_product(nums):
    max1 = max2 = 0
    print(f"khởi tạo: max1={max1}, max2={max2}")
    for x in nums:
        if x >= max1:
            branch = "x >= max1 → đẩy max1 cũ xuống max2"
            max2, max1 = max1, x
        elif x > max2:
            branch = "max2 < x < max1 → chỉ thay max2"
            max2 = x
        else:
            branch = "x <= max2 → bỏ qua"
        print(f"x={x}: {branch:38} | max1={max1}, max2={max2}")
    ans = (max1 - 1) * (max2 - 1)
    print(f"đáp án = ({max1}-1)*({max2}-1) = {ans}")
    return ans

trace_max_product([3, 4, 5, 2])

khởi tạo: max1=0, max2=0
x=3: x >= max1 → đẩy max1 cũ xuống max2     | max1=3, max2=0
x=4: x >= max1 → đẩy max1 cũ xuống max2     | max1=4, max2=3
x=5: x >= max1 → đẩy max1 cũ xuống max2     | max1=5, max2=4
x=2: x <= max2 → bỏ qua                     | max1=5, max2=4
đáp án = (5-1)*(4-1) = 12


12

### 8.1. Trace có đủ ba nhánh và giá trị trùng — `nums = [1, 5, 4, 5]` và `[2, 9, 7, 8, 1]`

In [14]:
trace_max_product([1, 5, 4, 5])
print()
trace_max_product([2, 9, 7, 8, 1])

khởi tạo: max1=0, max2=0
x=1: x >= max1 → đẩy max1 cũ xuống max2     | max1=1, max2=0
x=5: x >= max1 → đẩy max1 cũ xuống max2     | max1=5, max2=1
x=4: max2 < x < max1 → chỉ thay max2        | max1=5, max2=4
x=5: x >= max1 → đẩy max1 cũ xuống max2     | max1=5, max2=5
đáp án = (5-1)*(5-1) = 16

khởi tạo: max1=0, max2=0
x=2: x >= max1 → đẩy max1 cũ xuống max2     | max1=2, max2=0
x=9: x >= max1 → đẩy max1 cũ xuống max2     | max1=9, max2=2
x=7: max2 < x < max1 → chỉ thay max2        | max1=9, max2=7
x=8: max2 < x < max1 → chỉ thay max2        | max1=9, max2=8
x=1: x <= max2 → bỏ qua                     | max1=9, max2=8
đáp án = (9-1)*(8-1) = 56


56

Ở mảng thứ hai: `7` vào nhánh "chỉ thay max2" (vượt 2 nhưng không vượt 9), `8` lại thay `7`, còn `1` bị bỏ qua.

## 9. Cách 5 — `heapq.nlargest`

In [15]:
def max_product_heap(nums):
    a, c = heapq.nlargest(2, nums)
    return (a - 1) * (c - 1)

print(max_product_heap([3, 4, 5, 2]), max_product_heap([1, 5, 4, 5]), max_product_heap([3, 7]))

12 16 12


Cách này tổng quát nhất: muốn tích của top `k` số chỉ cần đổi `2` thành `k`.

## 10. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def maxProduct(self, nums: list[int]) -> int:
        max1 = max2 = 0
        for x in nums:
            if x >= max1:
                max1, max2 = x, max1
            elif x > max2:
                max2 = x
        return (max1 - 1) * (max2 - 1)

sol = Solution()
print(sol.maxProduct([3, 4, 5, 2]))   # 12
print(sol.maxProduct([1, 5, 4, 5]))   # 16
print(sol.maxProduct([3, 7]))         # 12

12
16
12


Bản ngắn (sắp xếp), cũng Accepted:

```python
class Solution:
    def maxProduct(self, nums: list[int]) -> int:
        nums = sorted(nums)
        return (nums[-1] - 1) * (nums[-2] - 1)
```

## 11. Vì sao thuật toán đúng?

**Bước 1 — Đáp án là hai số lớn nhất.** Gọi `a >= b` là hai giá trị lớn nhất (ở hai chỉ số khác nhau). Với cặp bất kỳ `(p, q)`, `p >= q`, ta có `p <= a` và `q <= b` (vì `q` là phần tử lớn thứ hai trong cặp, không thể vượt giá trị lớn thứ hai của cả mảng). Vì mọi thừa số `>= 0`: `(p-1)(q-1) <= (a-1)(q-1) <= (a-1)(b-1)`. ✓

**Bước 2 — Cách 4 tìm đúng `a`, `b`.** **Invariant:** sau khi xử lý `t` phần tử đầu, `max1` và `max2` là giá trị lớn nhất và lớn thứ hai (theo chỉ số khác nhau) trong `t` phần tử đó (coi phần còn thiếu là `0` khi `t < 2`).
- Gặp `x >= max1`: `x` thành lớn nhất, `max1` cũ thành lớn thứ hai (mọi phần tử khác `<= max2 <= max1` cũ). ✓
- Gặp `max2 < x < max1`: `max1` giữ nguyên, `x` thay `max2`. ✓
- Gặp `x <= max2`: top 2 không đổi. ✓

Vì `n >= 2` nên cuối cùng `max1`, `max2` đều là phần tử thật của mảng.

## 12. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(n²)` | `O(1)` | theo hint |
| 2. Sắp xếp | `O(n log n)` | `O(n)` (`sorted` tạo list mới) | ngắn |
| 3. Hai lượt | `O(n)` | `O(1)` | 3 lần duyệt (`max`, `index`, `max`) |
| 4. `max1/max2` | `O(n)` | `O(1)` | **khuyên dùng**, một lượt |
| 5. `heapq.nlargest` | `O(n log k)`, `k=2` | `O(k)` | tổng quát cho top-k |

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.maxProduct([3, 4, 5, 2]), 12)
check("ví dụ 2 (max trùng)", sol.maxProduct([1, 5, 4, 5]), 16)
check("ví dụ 3 (n = 2)", sol.maxProduct([3, 7]), 12)
check("toàn số 1", sol.maxProduct([1, 1, 1]), 0)
check("có số 1 và số lớn", sol.maxProduct([1, 1000]), 0)
check("hai giá trị lớn nhất 1000", sol.maxProduct([1000, 1, 1000]), 999 * 999)
check("max ở đầu, max2 ở cuối", sol.maxProduct([10, 1, 2, 3, 9]), 9 * 8)
check("giảm dần", sol.maxProduct([9, 8, 7, 6]), 8 * 7)

PASS  ví dụ 1: 12
PASS  ví dụ 2 (max trùng): 16
PASS  ví dụ 3 (n = 2): 12
PASS  toàn số 1: 0
PASS  có số 1 và số lớn: 0
PASS  hai giá trị lớn nhất 1000: 998001
PASS  max ở đầu, max2 ở cuối: 72
PASS  giảm dần: 56


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

Thêm một biến thể của cách 4 dùng `>` thay vì `>=` để xác nhận nhận xét ở mục 7.

In [18]:
import random

def max_product_one_pass_strict(nums):
    max1 = max2 = 0
    for x in nums:
        if x > max1:
            max1, max2 = x, max1
        elif x > max2:
            max2 = x
    return (max1 - 1) * (max2 - 1)

rng = random.Random(1464)
methods = [max_product_brute, max_product_sort, max_product_two_pass, max_product_one_pass,
           max_product_heap, max_product_one_pass_strict, Solution().maxProduct]
for _ in range(1000):
    n = rng.randint(2, 60)
    hi = rng.choice([3, 10, 1000])                 # hi nhỏ → nhiều giá trị trùng
    nums = [rng.randint(1, hi) for _ in range(n)]
    results = {f(nums) for f in methods}
    assert len(results) == 1, (nums, results)
print("1000 mảng ngẫu nhiên: 7 hàm cho cùng kết quả")

1000 mảng ngẫu nhiên: 7 hàm cho cùng kết quả


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Loại bỏ theo **giá trị** thay vì theo **chỉ số**

In [19]:
def wrong_remove_all_max(nums):
    m1 = max(nums)
    rest = [x for x in nums if x != m1]     # bỏ MỌI số bằng m1
    m2 = max(rest) if rest else m1
    return (m1 - 1) * (m2 - 1)

print("Sai:", wrong_remove_all_max([1, 5, 4, 5]), "| Đúng:", max_product_one_pass([1, 5, 4, 5]))

Sai: 12 | Đúng: 16


**Vì sao sai:** hai số 5 ở hai chỉ số khác nhau là cặp hợp lệ. **Sửa:** bỏ đúng một chỉ số (`k != i1`) hoặc dùng `max1/max2`.

### Lỗi 2 — Đẩy xuống sai thứ tự

In [20]:
def wrong_order(nums):
    max1 = max2 = 0
    for x in nums:
        if x >= max1:
            max1 = x
            max2 = max1          # max1 đã bị ghi đè → max2 = x
        elif x > max2:
            max2 = x
    return (max1 - 1) * (max2 - 1)

print("Sai:", wrong_order([3, 4, 5, 2]), "| Đúng:", max_product_one_pass([3, 4, 5, 2]))

Sai: 16 | Đúng: 12


**Sửa:** `max2 = max1` trước, rồi `max1 = x`; hoặc gán đồng thời `max1, max2 = x, max1`.

### Lỗi 3 — Quên nhánh `elif` (chỉ cập nhật khi vượt `max1`)

In [21]:
def wrong_no_elif(nums):
    max1 = max2 = 0
    for x in nums:
        if x > max1:
            max1, max2 = x, max1
    return (max1 - 1) * (max2 - 1)

print("Sai:", wrong_no_elif([9, 2, 8]), "| Đúng:", max_product_one_pass([9, 2, 8]))

Sai: -8 | Đúng: 56


**Vì sao sai:** `8` đến sau `9` nên không vượt `max1`, nhưng nó phải thay `max2 = 2`.

### Lỗi 4 — Quên `- 1`, hoặc trừ 1 sau khi nhân

In [22]:
a, c = 5, 4
print("a*c - 1      =", a * c - 1)
print("(a-1)*(c-1)  =", (a - 1) * (c - 1), "← đúng đề")

a*c - 1      = 19
(a-1)*(c-1)  = 12 ← đúng đề


### Lỗi 5 — Dùng `nums.sort()` rồi tiếp tục dùng `nums` ở chỗ khác

`nums.sort()` sửa mảng của người gọi. Trên LeetCode không sao; trong code thật (hay trong notebook test nhiều hàm trên cùng dữ liệu) sẽ gây bất ngờ. Ưu tiên `sorted(nums)` khi không chắc.

## 15. Mẹo nhận dạng pattern

- "Chọn 2 phần tử để **tối đa** một biểu thức **đơn điệu**" → chỉ cần **top-2** (hoặc bottom-2) → một lượt `max1/max2`.
- Khi có **số âm**, tích có thể lớn nhất từ **hai số nhỏ nhất** → giữ cả `min1/min2`.
- "Top-k" tổng quát → `heapq.nlargest(k, ...)` hoặc heap kích thước `k`.
- Bài cùng họ: *Maximum Product of Three Numbers* (628), *Maximum Product Difference Between Two Pairs* (1913), *Third Maximum Number* (414), *Kth Largest Element in an Array* (215).

## 16. Biến thể

### 16.1. Tích lớn nhất của hai phần tử khi có số âm

In [23]:
def max_pair_product_with_negatives(nums):
    s = sorted(nums)
    return max(s[-1] * s[-2], s[0] * s[1])

print(max_pair_product_with_negatives([-10, -9, 1, 2]))   # 90
print(max_pair_product_with_negatives([-1, 5, 3]))        # 15

90
15


### 16.2. Maximum Product Difference Between Two Pairs (LeetCode 1913)
Chọn 4 chỉ số khác nhau `w, x, y, z`, tối đa `nums[w]*nums[x] - nums[y]*nums[z]` (mọi số dương) → top-2 nhân nhau trừ bottom-2 nhân nhau.

In [24]:
def max_product_difference(nums):
    s = sorted(nums)
    return s[-1] * s[-2] - s[0] * s[1]

print(max_product_difference([5, 6, 2, 7, 4]))   # 34 = 7*6 - 2*4

34


### 16.3. Trả về **chỉ số** của hai phần tử được chọn

In [25]:
def max_product_indices(nums):
    i1 = i2 = -1
    for k, x in enumerate(nums):
        if i1 == -1 or x >= nums[i1]:
            i1, i2 = k, i1
        elif i2 == -1 or x > nums[i2]:
            i2 = k
    return sorted([i1, i2])

print(max_product_indices([3, 4, 5, 2]))   # [1, 2]
print(max_product_indices([1, 5, 4, 5]))   # [1, 3]

[1, 2]
[1, 3]


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** Lập bảng `max1/max2` cho `nums = [6, 6, 2, 7, 3]`. Đáp án?

**Bài 2 — Dự đoán output.**
```python
a, c = 1, 2
a, c = c, a
print(a, c, sorted([5, 1, 5])[-2], heapq.nlargest(2, [4, 9, 9]))
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải một lượt

In [26]:
def exercise_max_product(nums):
    # TODO 1: max1 = max2 = 0
    # TODO 2: duyệt x: nhánh đẩy xuống / nhánh thay max2
    # TODO 3: return (max1 - 1) * (max2 - 1)
    pass

# assert exercise_max_product([3,4,5,2]) == 12
# assert exercise_max_product([1,5,4,5]) == 16

### Bài 4 — TODO: Third Maximum Number (LeetCode 414)
Trả **giá trị phân biệt** lớn thứ ba; nếu không có (ít hơn 3 giá trị phân biệt) trả giá trị lớn nhất.

In [27]:
def exercise_third_max(nums):
    # TODO: giữ m1 > m2 > m3 (PHÂN BIỆT), bỏ qua x bằng một trong ba
    pass

# assert exercise_third_max([3,2,1]) == 1
# assert exercise_third_max([1,2]) == 2
# assert exercise_third_max([2,2,3,1]) == 1

### Bài 5 — TODO: Maximum Product of Three Numbers (LeetCode 628), một lượt
Có số âm. Trả tích lớn nhất của 3 phần tử ở 3 chỉ số khác nhau, trong `O(n)` không sắp xếp.

In [28]:
def exercise_max_product_three(nums):
    # TODO: giữ 3 số lớn nhất và 2 số nhỏ nhất
    #       đáp án = max(max1*max2*max3, max1*min1*min2)
    pass

# assert exercise_max_product_three([1,2,3]) == 6
# assert exercise_max_product_three([-10,-10,1,3,2]) == 300
# assert exercise_max_product_three([-1,-2,-3]) == -6

### Bài 6 — Mini challenge: tích lớn nhất của hai phần tử **không kề nhau** (`|i - j| >= 2`)
Giá trị như đề gốc (`>= 1`), biểu thức `(nums[i]-1)*(nums[j]-1)`, `n >= 3`. Làm trong `O(n)`.

In [29]:
def exercise_max_product_non_adjacent(nums):
    # TODO: duyệt j từ 2; giữ best_left = max(nums[0..j-2]); cập nhật đáp án với nums[j]
    pass

# assert exercise_max_product_non_adjacent([1, 9, 9, 1]) == 0
# assert exercise_max_product_non_adjacent([5, 9, 9, 6]) == 8 * 5

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Chỉ hai số lớn nhất quan trọng. | `if x >= max1 ... elif x > max2`. | `max1, max2 = x, max1` |
| 4 | Giống `max1/max2` nhưng 3 hạng và **phân biệt**. | khởi tạo `None` hoặc `-inf`; `if x in (m1, m2, m3): continue`. | đẩy xuống dây chuyền: `m1, m2, m3 = x, m1, m2` |
| 5 | Hai số âm nhân nhau thành dương. | 5 biến: `max1..3`, `min1..2`. | `max(max1*max2*max3, max1*min1*min2)` |
| 6 | Với mỗi `j`, cặp tốt nhất là `nums[j]` với max bên trái cách ít nhất 2. | biến `best_left` cập nhật bằng `nums[j-2]`. | `best_left = max(best_left, nums[j-2]); ans = max(ans, (best_left-1)*(nums[j]-1))` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `x=6`: (6,0); `x=6`: (6,6); `x=2`: bỏ qua; `x=7`: (7,6); `x=3`: bỏ qua → `(7-1)*(6-1) = 30`.
**Bài 2:** `2 1 5 [9, 9]`.

In [30]:
def answer_max_product(nums):
    max1 = max2 = 0
    for x in nums:
        if x >= max1:
            max1, max2 = x, max1
        elif x > max2:
            max2 = x
    return (max1 - 1) * (max2 - 1)

def answer_third_max(nums):
    m1 = m2 = m3 = None
    for x in nums:
        if x in (m1, m2, m3):
            continue
        if m1 is None or x > m1:
            m1, m2, m3 = x, m1, m2
        elif m2 is None or x > m2:
            m2, m3 = x, m2
        elif m3 is None or x > m3:
            m3 = x
    return m1 if m3 is None else m3

def answer_max_product_three(nums):
    INF = float("inf")
    max1 = max2 = max3 = -INF
    min1 = min2 = INF
    for x in nums:
        if x >= max1:
            max1, max2, max3 = x, max1, max2
        elif x >= max2:
            max2, max3 = x, max2
        elif x > max3:
            max3 = x
        if x <= min1:
            min1, min2 = x, min1
        elif x < min2:
            min2 = x
    return max(max1 * max2 * max3, max1 * min1 * min2)

def answer_max_product_non_adjacent(nums):
    best_left, ans = 0, 0
    for j in range(2, len(nums)):
        best_left = max(best_left, nums[j - 2])
        ans = max(ans, (best_left - 1) * (nums[j] - 1))
    return ans

assert answer_max_product([3,4,5,2]) == 12
assert answer_max_product([1,5,4,5]) == 16
assert answer_third_max([3,2,1]) == 1
assert answer_third_max([1,2]) == 2
assert answer_third_max([2,2,3,1]) == 1
assert answer_max_product_three([1,2,3]) == 6
assert answer_max_product_three([-10,-10,1,3,2]) == 300
assert answer_max_product_three([-1,-2,-3]) == -6
assert answer_max_product_non_adjacent([1, 9, 9, 1]) == 0
assert answer_max_product_non_adjacent([5, 9, 9, 6]) == 8 * 5
from itertools import combinations
for _ in range(500):
    arr = [rng.randint(-10, 10) for _ in range(rng.randint(3, 9))]
    assert answer_max_product_three(arr) == max(a * b * c for a, b, c in combinations(arr, 3)), arr
    pos = [rng.randint(1, 20) for _ in range(rng.randint(3, 9))]
    assert answer_max_product_non_adjacent(pos) == max((pos[i]-1)*(pos[j]-1) for i in range(len(pos)) for j in range(i+2, len(pos)))
    assert answer_third_max(arr) == (sorted(set(arr))[-3] if len(set(arr)) >= 3 else max(arr))
print("Bài 1:", answer_max_product([6, 6, 2, 7, 3]))
a, c = 1, 2
a, c = c, a
print("Bài 2:", a, c, sorted([5, 1, 5])[-2], heapq.nlargest(2, [4, 9, 9]))
print("Đáp án bài 3–6: PASS (kể cả đối chiếu vét cạn)")

Bài 1: 30
Bài 2: 2 1 5 [9, 9]
Đáp án bài 3–6: PASS (kể cả đối chiếu vét cạn)


## 20. Template nhớ nhanh + checklist submit

```python
max1 = max2 = GIÁ_TRỊ_NHỎ_HƠN_MỌI_PHẦN_TỬ
for x in nums:
    if x >= max1:
        max1, max2 = x, max1      # đẩy quán quân cũ xuống hạng 2
    elif x > max2:
        max2 = x                  # chỉ thay hạng 2
# dùng max1, max2
```

**Checklist trước khi nộp**
- [ ] Hai **chỉ số** khác nhau — giá trị bằng nhau vẫn được (không loại theo giá trị).
- [ ] Thứ tự đẩy xuống: `max2` nhận `max1` cũ **trước** khi `max1` đổi.
- [ ] Có nhánh `elif x > max2`.
- [ ] Biểu thức là `(a - 1) * (b - 1)`.
- [ ] Lập luận "hai số lớn nhất" chỉ đúng vì mọi `nums[i] - 1 >= 0`.
- [ ] Method tên `maxProduct`, trả về `int`.